# VisionPort – Intelligent Vehicle Detection and Traffic Monitoring

**YOLOv8 + OpenCV | Jupyter Notebook | CPU-friendly starter implementation**

This notebook extends the supplied image-only demo into a traffic-video monitoring workflow. It covers vehicle detection, labels/confidence, vehicle counting, traffic-density categories, congestion alerts, tracking IDs, approximate motion/speed, signal-colour estimation, waiting-time estimates, line-crossing counts, wrong-direction flags and restricted-zone alerts.

> **Important limitations:** Speed requires a real-world distance calibration. Signal colour detection requires selecting a traffic-light region. Entry/exit and wrong-way logic require configuring a virtual line and expected direction. Illegal-parking alerts require a restricted-zone polygon. Triple-riding detection is not reliably supported by the standard COCO YOLO model; it needs a custom trained model and suitable camera view. These are marked in the relevant sections.


## 1. Install dependencies
Run once. Restart the kernel if Jupyter asks you to.

In [ ]:
%pip install -q ultralytics opencv-python matplotlib pandas ipywidgets pillow

## 2. Import libraries and load YOLOv8

In [ ]:
from ultralytics import YOLO
from IPython.display import display, Video
import ipywidgets as widgets
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from pathlib import Path
import time, math, os

MODEL_NAME = "yolov8n.pt"   # Lightweight pretrained COCO model
CONFIDENCE = 0.25
IMAGE_SIZE = 640
DEVICE = "cpu"              # Change to 0 if a compatible CUDA GPU is available

model = YOLO(MODEL_NAME)
print("YOLO model loaded:", MODEL_NAME)
print("Vehicle classes:", {2:"car", 3:"motorcycle", 5:"bus", 7:"truck"})


## 3. Upload a traffic image (Tasks 2–3, 6–9)
Choose a JPG/PNG image. This image workflow is useful for still-image demonstrations; the video workflow follows.

In [ ]:
image_upload = widgets.FileUpload(accept=".jpg,.jpeg,.png,.webp", multiple=False)
display(image_upload)

def uploaded_bytes(widget):
    value = widget.value
    if not value:
        raise ValueError("Upload a file first.")
    if isinstance(value, dict):
        info = value if "content" in value else next(iter(value.values()))
    else:
        info = value[0]
    return bytes(info["content"]), info.get("name", "upload")



In [ ]:
img_bytes, img_name = uploaded_bytes(image_upload)
image = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
if image is None:
    raise ValueError("Could not read image. Please upload a valid image.")
plt.figure(figsize=(12,6))
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(img_name)
plt.show()


## 4. Detect and count vehicles in the image

In [ ]:
image_result = model.predict(image, conf=CONFIDENCE, imgsz=IMAGE_SIZE, device=DEVICE, verbose=False)[0]
vehicle_names = {"car", "motorcycle", "bus", "truck"}
image_rows = []
if image_result.boxes is not None:
    for b in image_result.boxes:
        name = image_result.names[int(b.cls.item())]
        if name in vehicle_names:
            image_rows.append({
                "vehicle": name,
                "confidence": round(float(b.conf.item()), 3),
                "x1": round(float(b.xyxy[0][0].item()), 1),
                "y1": round(float(b.xyxy[0][1].item()), 1),
                "x2": round(float(b.xyxy[0][2].item()), 1),
                "y2": round(float(b.xyxy[0][3].item()), 1)
            })
image_df = pd.DataFrame(image_rows)
display(image_df)
print("Total vehicles:", len(image_df))
print("Counts by type:")
display(image_df["vehicle"].value_counts().rename_axis("vehicle").reset_index(name="count"))
annotated = image_result.plot()
plt.figure(figsize=(12,7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
cv2.imwrite("visionport_image_result.jpg", annotated)
image_df.to_csv("visionport_image_detections.csv", index=False)
print("Saved image result and CSV.")


## 5. Upload and play a traffic video (Tasks 4–5)

Upload an MP4, AVI or MOV file. The notebook reads frames with OpenCV. A few frames are saved as sample images.

In [ ]:
video_upload = widgets.FileUpload(accept=".mp4,.avi,.mov,.mkv", multiple=False)
display(video_upload)


In [ ]:
video_bytes, video_name = uploaded_bytes(video_upload)
input_video = "visionport_input_video" + Path(video_name).suffix.lower()
with open(input_video, "wb") as f:
    f.write(video_bytes)

cap = cv2.VideoCapture(input_video)
if not cap.isOpened():
    raise ValueError("Could not open video. Try an MP4 encoded with H.264.")
fps = cap.get(cv2.CAP_PROP_FPS) or 25
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {width}x{height}, {fps:.2f} FPS, {total_frames} frames")
ok, first_frame = cap.read()
if ok:
    plt.figure(figsize=(12,6))
    plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.show()
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
os.makedirs("visionport_frames", exist_ok=True)
for i in range(5):
    ok, frame = cap.read()
    if not ok: break
    cv2.imwrite(f"visionport_frames/frame_{i:03d}.jpg", frame)
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)


## 6. Configure the traffic-scene measurements

- `COUNT_LINE_Y`: horizontal virtual line used for entry/exit counting.
- `EXPECTED_DIRECTION`: expected vertical movement across the image (`down` or `up`); adjust to your camera.
- `DENSITY_MEDIUM` and `DENSITY_HIGH`: example thresholds; tune for the camera.
- `RESTRICTED_ZONE`: optional polygon in pixel coordinates. Leave empty to disable.
- `METERS_PER_PIXEL`: set only if you have calibrated the scene using a known road distance. Otherwise speed is reported as pixels/second, not km/h.

In [ ]:
COUNT_LINE_Y = int(height * 0.60)
EXPECTED_DIRECTION = "down"  # "down" or "up"
DENSITY_MEDIUM = 8
DENSITY_HIGH = 15
CONGESTION_THRESHOLD = 12
CONGESTION_MIN_SECONDS = 3

# Example restricted polygon; replace coordinates for your own frame, or set [] to disable.
RESTRICTED_ZONE = []  # e.g. [(100,300), (500,300), (550,600), (80,600)]

# Calibration: enter measured metres per pixel only after calibrating your camera.
METERS_PER_PIXEL = None


## 7. Run video detection, tracking, counting and alerts (Tasks 6–12, 16, 18–19)

YOLO tracking assigns IDs across frames. Counts are based on vehicles currently visible. Line-crossing counts are approximate and depend on the selected line and camera angle. Wrong-way alerts compare the movement of each tracked centre with the configured expected direction.

In [ ]:
cap = cv2.VideoCapture(input_video)
out_path = "visionport_annotated_video.mp4"
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

track_history = {}
crossed_ids = set()
entry_count = 0
exit_count = 0
wrong_way_ids = set()
frame_logs = []
vehicle_wait_start = {}
last_positions = {}
congestion_start = None
congestion_alerts = 0

frame_index = 0
MAX_FRAMES = None  # Set e.g. 1000 for a quicker test; None processes the full video.

while cap.isOpened():
    ok, frame = cap.read()
    if not ok: break
    frame_index += 1
    if MAX_FRAMES is not None and frame_index > MAX_FRAMES: break
    timestamp = frame_index / fps

    tracked = model.track(frame, persist=True, conf=CONFIDENCE, imgsz=IMAGE_SIZE,
                          device=DEVICE, classes=[2,3,5,7], tracker="bytetrack.yaml",
                          verbose=False)[0]
    current_count = 0
    if tracked.boxes is not None and tracked.boxes.id is not None:
        boxes = tracked.boxes.xyxy.cpu().numpy()
        ids = tracked.boxes.id.int().cpu().tolist()
        clss = tracked.boxes.cls.int().cpu().tolist()
        confs = tracked.boxes.conf.cpu().tolist()

        for box, tid, cls_id, conf in zip(boxes, ids, clss, confs):
            x1,y1,x2,y2 = map(int, box)
            cx,cy = (x1+x2)//2, (y1+y2)//2
            name = tracked.names[int(cls_id)]
            current_count += 1

            prev = last_positions.get(tid)
            if prev is not None:
                px, py, prev_t = prev
                dy = cy - py
                if EXPECTED_DIRECTION == "down" and dy < -3:
                    wrong_way_ids.add(tid)
                elif EXPECTED_DIRECTION == "up" and dy > 3:
                    wrong_way_ids.add(tid)

                # Approximate speed. Pixel speed is always available; km/h needs calibration.
                dt = max(timestamp - prev_t, 1/fps)
                pixel_speed = math.hypot(cx-px, cy-py) / dt
            else:
                pixel_speed = 0.0
            last_positions[tid] = (cx, cy, timestamp)

            # Estimate stationary/waiting duration using a small movement threshold.
            if prev is not None and math.hypot(cx-prev[0], cy-prev[1]) < 4:
                vehicle_wait_start.setdefault(tid, timestamp)
            else:
                vehicle_wait_start[tid] = timestamp
            wait_seconds = timestamp - vehicle_wait_start.get(tid, timestamp)

            # Count each tracked vehicle once when its centre crosses the virtual line.
            if tid not in crossed_ids and prev is not None:
                crossed = (prev[1] < COUNT_LINE_Y <= cy) if EXPECTED_DIRECTION == "down" else (prev[1] > COUNT_LINE_Y >= cy)
                if crossed:
                    entry_count += 1
                    crossed_ids.add(tid)

            color = (0, 0, 255) if tid in wrong_way_ids else (0, 220, 0)
            label = f"{name} ID:{tid} {conf:.2f}"
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, label, (x1, max(20,y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
            cv2.putText(frame, f"wait~{wait_seconds:.1f}s speed~{pixel_speed:.1f}px/s",
                        (x1, min(height-8,y2+18)), cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

            frame_logs.append({"time_s": round(timestamp,2), "track_id":tid, "vehicle":name,
                               "confidence":round(float(conf),3), "center_x":cx, "center_y":cy,
                               "speed_pixels_s":round(pixel_speed,2), "waiting_estimate_s":round(wait_seconds,2),
                               "wrong_way":tid in wrong_way_ids})

    # Traffic density and congestion from visible vehicle count.
    density = "Low" if current_count < DENSITY_MEDIUM else ("Medium" if current_count < DENSITY_HIGH else "High")
    if current_count >= CONGESTION_THRESHOLD:
        if congestion_start is None: congestion_start = timestamp
        elif timestamp - congestion_start >= CONGESTION_MIN_SECONDS:
            congestion_alerts += 1
            cv2.putText(frame, "CONGESTION ALERT", (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
    else:
        congestion_start = None

    cv2.line(frame, (0, COUNT_LINE_Y), (width, COUNT_LINE_Y), (255,0,0), 2)
    if RESTRICTED_ZONE:
        polygon = np.array(RESTRICTED_ZONE, np.int32)
        cv2.polylines(frame, [polygon], True, (0,165,255), 2)
        for box in (tracked.boxes.xyxy.cpu().numpy() if tracked.boxes is not None and len(tracked.boxes) else []):
            bx,by = int((box[0]+box[2])/2), int(box[3])
            if cv2.pointPolygonTest(polygon, (bx,by), False) >= 0:
                cv2.putText(frame, "RESTRICTED ZONE", (bx,by), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,0,255), 2)

    cv2.putText(frame, f"Vehicles: {current_count} | Density: {density}", (20,35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255,255,0), 2)
    cv2.putText(frame, f"Line crossings: {entry_count} | Wrong-way IDs: {len(wrong_way_ids)}", (20,62),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,255,0), 2)
    writer.write(frame)

cap.release()
writer.release()
print("Processed frames:", frame_index)
print("Line crossings (configured direction):", entry_count)
print("Wrong-way tracked IDs:", sorted(wrong_way_ids))
print("Congestion alert frames:", congestion_alerts)
print("Annotated video saved:", out_path)


## 8. View the processed traffic video

In [ ]:
display(Video(out_path, embed=True, width=800))

## 9. Export tracking and traffic reports (Tasks 8–10, 12–13, 15–16, 18)

The report contains per-frame tracked detections. A track may appear in many rows, so do not sum rows to calculate unique vehicles.

In [ ]:
tracking_df = pd.DataFrame(frame_logs)
tracking_df.to_csv("visionport_tracking_report.csv", index=False)
if not tracking_df.empty:
    unique_vehicles = tracking_df.groupby("track_id")["vehicle"].first().value_counts()
    print("Unique tracked vehicles by type:")
    display(unique_vehicles.rename_axis("vehicle").reset_index(name="unique_tracks"))
    print("Average estimated waiting time (tracked observations):",
          round(tracking_df["waiting_estimate_s"].mean(), 2), "seconds")
    print("Maximum simultaneous vehicle count is estimated from annotated video overlay.")
display(tracking_df.head(20))
print("Saved: visionport_tracking_report.csv")


## 10. Traffic signal status (Task 14) – optional ROI colour heuristic

Select a rectangular region containing the traffic light in the original video frame. The HSV colour heuristic below is only a demonstration: sunlight, reflections, LED flicker and nearby objects can cause errors. A trained traffic-light classifier is more reliable.

In [ ]:
# Set ROI coordinates based on the video frame. Example only; replace with actual values.
SIGNAL_ROI = None  # e.g. (x1, y1, x2, y2)

def estimate_signal_status(frame, roi):
    if roi is None:
        return "ROI_NOT_CONFIGURED"
    x1,y1,x2,y2 = roi
    crop = frame[y1:y2, x1:x2]
    if crop.size == 0: return "INVALID_ROI"
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    red1 = cv2.inRange(hsv, (0,100,80), (10,255,255))
    red2 = cv2.inRange(hsv, (160,100,80), (180,255,255))
    yellow = cv2.inRange(hsv, (15,80,100), (40,255,255))
    green = cv2.inRange(hsv, (40,70,70), (90,255,255))
    scores = {"RED": int(cv2.countNonZero(red1)+cv2.countNonZero(red2)),
              "YELLOW": int(cv2.countNonZero(yellow)),
              "GREEN": int(cv2.countNonZero(green))}
    best = max(scores, key=scores.get)
    return best if scores[best] > 2 else "UNKNOWN"

print("Configure SIGNAL_ROI and call estimate_signal_status(frame, SIGNAL_ROI) on frames.")


## 11. Speed estimation and entry/exit counts (Tasks 13 and 16)

The tracking report currently records speed in **pixels/second**. Convert to km/h only after measuring a known distance in the same image plane:

`speed_kmh = pixels_per_second × metres_per_pixel × 3.6`

A single horizontal line counts crossings in the configured direction. To separately count entering and leaving traffic, define two directional lines or configure the camera view and logic for your junction. The simple counter above should be described as an approximate line-crossing count, not a complete junction flow model.

In [ ]:
if METERS_PER_PIXEL is not None and not tracking_df.empty:
    tracking_df["speed_kmh_approx"] = tracking_df["speed_pixels_s"] * METERS_PER_PIXEL * 3.6
    tracking_df.to_csv("visionport_tracking_report_calibrated.csv", index=False)
    display(tracking_df[["track_id","speed_pixels_s","speed_kmh_approx"]].head())
else:
    print("No real-world speed shown: set METERS_PER_PIXEL after camera calibration.")


## 12. Triple-riding detection (Task 20) – model limitation

The pretrained `yolov8n.pt` COCO model detects motorcycles and people as separate classes, but it does not reliably determine how many people are riding one motorcycle. Do not report a guessed count as a confirmed triple-riding violation. For this task, train or obtain a dedicated model with labels such as `two_person_motorcycle` and `three_person_motorcycle`, using labelled images from an appropriate camera angle, then load that custom weights file with `YOLO("your_custom_model.pt")`.

## 13. Task coverage checklist

| Assignment task | Coverage in this notebook |
|---|---|
| 1. Install Python/OpenCV/libraries | Install cell; Python/Jupyter must already be installed |
| 2. Capture traffic image using camera | Upload workflow; live camera capture can be added with `cv2.VideoCapture(0)` |
| 3. Read/display traffic images | Implemented |
| 4. Load/play traffic video | Upload and playback |
| 5. Extract video frames | Saves sample frames |
| 6. Detect cars using YOLO | Implemented |
| 7. Detect bikes, buses, trucks | Implemented (COCO classes) |
| 8. Names and confidence | Implemented |
| 9. Vehicle count | Implemented |
| 10. Traffic density | Low/Medium/High thresholds |
| 11. Congestion detection | Threshold and duration alert |
| 12. Track vehicles | ByteTrack IDs |
| 13. Approximate speed | Pixels/second; km/h only after calibration |
| 14. Signal status | Optional ROI colour heuristic |
| 15. Waiting time | Approximate stationary-time estimate |
| 16. Entering/leaving count | Configured virtual-line crossing; full junction in/out needs extra line logic |
| 17. Wrong turns | Not reliably implemented; requires road-lane/intersection paths and turn-zone calibration |
| 18. Wrong-way movement | Basic direction heuristic |
| 19. Restricted-area parking | Optional polygon overlap alert; dwell-time rule can be added |
| 20. Triple riding | Requires custom trained model |

## 14. Output files

- `visionport_image_result.jpg`
- `visionport_image_detections.csv`
- `visionport_annotated_video.mp4`
- `visionport_tracking_report.csv`
- `visionport_frames/` sample frames

**For your project demonstration:** use a clear traffic video, configure the line and any ROI/polygon using the actual frame dimensions, and explain the limitations above honestly. Detection accuracy depends on camera quality, lighting, occlusion and model training.
